In [108]:
# Import required libraries
import pandas as pd
import os
import glob

In [109]:
# Define base path and years
BASE_PATH = "../data/raw"
YEARS = list(range(2015, 2023))

# Define table types and their categories
TABLE_TYPES = {
    "Table_1.2": "Total Accidental Deaths",
    "Table_1A.4": "Mode of Transportation",
    "Table_1A.5": "Month of Occurrence",
    "Table_1A.6": "Time of Occurrence",
    "Table_1A.7": "Road Classification",
    "Table_1A.11": "Place of Occurrence"
}

In [110]:
# Load all files and store in a dictionary
all_files = {}

for year in YEARS:
    all_files[year] = {}
    for table, category in TABLE_TYPES.items():
        file_path = os.path.join(BASE_PATH, str(year), f"ADSI_{year}_{table}.csv")
        if os.path.exists(file_path):
            try:
                df = pd.read_csv(file_path, encoding='utf-8')
                all_files[year][table] = {"df": df, "status": "loaded", "category": category}
            except UnicodeDecodeError:
                try:
                    df = pd.read_csv(file_path, encoding='latin1')
                    all_files[year][table] = {"df": df, "status": "loaded (latin1)", "category": category}
                except Exception as e:
                    all_files[year][table] = {"df": None, "status": f"error: {e}", "category": category}
        else:
            all_files[year][table] = {"df": None, "status": "file not found", "category": category}

print("All files loading attempt complete.")

All files loading attempt complete.


In [111]:
# Inspect one file per table type (using 2020 as reference)
REFERENCE_YEAR = 2020

for table, category in TABLE_TYPES.items():
    print(f"\n{'='*60}")
    print(f"Category : {category}")
    print(f"Table    : {table} | Year: {REFERENCE_YEAR}")
    print(f"{'='*60}")
    
    entry = all_files[REFERENCE_YEAR][table]
    if entry["df"] is not None:
        df = entry["df"]
        print(f"Shape    : {df.shape}")
        print(f"Columns  : {list(df.columns)}")
        print(f"\nFirst 5 rows:")
        print(df.head())
    else:
        print(f"Status   : {entry['status']}")


Category : Total Accidental Deaths
Table    : Table_1.2 | Year: 2020
Shape    : (93, 9)
Columns  : ['Si. No. (Col. 1)', 'Category', 'State/UT/City (Col. 2)', 'Total Number of Accidental Deaths - Forces of Nature (Col. 3)', 'Total Number of Accidental Deaths - Other Causes (Col. 4)', 'Total Number of Accidental Deaths - Total (Col. 5)', 'Percentage Share in Total Deaths (Col. 6)', 'Projected Mid-Year Population (in Lakh) (Col. 7)', 'Rate of Accidental Deaths (Col. 8) = (Col.5/ Col.7)']

First 5 rows:
  Si. No. (Col. 1) Category State/UT/City (Col. 2)  Total Number of Accidental Deaths - Forces of Nature (Col. 3)  Total Number of Accidental Deaths - Other Causes (Col. 4)  Total Number of Accidental Deaths - Total (Col. 5)  Percentage Share in Total Deaths (Col. 6)  Projected Mid-Year Population (in Lakh) (Col. 7)  Rate of Accidental Deaths (Col. 8) = (Col.5/ Col.7)
0                1    State         Andhra Pradesh                                                164                      

In [129]:
# Standard State/UT names mapping
STATE_UT_NAMES = {
    # States
    "ANDHRA PRADESH": "Andhra Pradesh",
    "ARUNACHAL PRADESH": "Arunachal Pradesh",
    "ASSAM": "Assam",
    "BIHAR": "Bihar",
    "CHHATTISGARH": "Chhattisgarh",
    "GOA": "Goa",
    "GUJARAT": "Gujarat",
    "HARYANA": "Haryana",
    "HIMACHAL PRADESH": "Himachal Pradesh",
    "JHARKHAND": "Jharkhand",
    "KARNATAKA": "Karnataka",
    "KERALA": "Kerala",
    "MADHYA PRADESH": "Madhya Pradesh",
    "MAHARASHTRA": "Maharashtra",
    "MANIPUR": "Manipur",
    "MEGHALAYA": "Meghalaya",
    "MIZORAM": "Mizoram",
    "NAGALAND": "Nagaland",
    "ODISHA": "Odisha",
    "PUNJAB": "Punjab",
    "RAJASTHAN": "Rajasthan",
    "SIKKIM": "Sikkim",
    "TAMIL NADU": "Tamil Nadu",
    "TELANGANA": "Telangana",
    "TRIPURA": "Tripura",
    "UTTAR PRADESH": "Uttar Pradesh",
    "UTTARAKHAND": "Uttarakhand",
    "WEST BENGAL": "West Bengal",
    # UTs
    "ANDAMAN AND NICOBAR ISLANDS": "Andaman and Nicobar Islands",
    "ANDAMAN & NICOBAR ISLANDS": "Andaman and Nicobar Islands",
    "CHANDIGARH": "Chandigarh",
    "DADRA AND NAGAR HAVELI": "Dadra and Nagar Haveli and Daman and Diu",
    "DAMAN AND DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "DADRA AND NAGAR HAVELI AND DAMAN AND DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "DADRA & NAGAR HAVELI & DAMAN & DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "D & N HAVELI AND DAMAN & DIU": "Dadra and Nagar Haveli and Daman and Diu",
"ANDAMAND AND NICOBAR ISLANDS": "Andaman and Nicobar Islands",
    "DELHI": "Delhi",
    "JAMMU AND KASHMIR": "Jammu and Kashmir",
    "JAMMU & KASHMIR": "Jammu and Kashmir",
    "LADAKH": "Jammu and Kashmir",
    "LAKSHADWEEP": "Lakshadweep",
    "PUDUCHERRY": "Puducherry",
    "PONDICHERRY": "Puducherry",
}
print("State/UT name mapping defined.")

State/UT name mapping defined.


In [117]:
# Clean Table_1.2 - Total Accidental Deaths
all_years_data = []

for year in YEARS:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()
    
    # Find category column and name column
    cat_col = df.columns[1]
    name_col = df.columns[2]
    
    # 2019 has different structure — category col has state names instead of State/UT label
    if df[cat_col].str.strip().str.upper().isin(["STATE", "UT", "UNION TERRITORY"]).any():
        # Normal structure — 2015, 2016, 2017, 2018, 2020, 2021, 2022
        df = df[df[cat_col].str.strip().str.upper().isin(["STATE", "UT", "UNION TERRITORY"])]
    else:
        # 2019 structure — filter by known State/UT names
        df = df[df[name_col].str.strip().str.upper().isin(VALID_STATE_UT_KEYS)]
        # Swap: name is in col 2 (index 1) actually
        name_col = df.columns[1]

    # Find total deaths column dynamically
    total_col = None
    for col in df.columns:
        if "Total" in col and ("Col. 5" in col or "Col.5" in col or "Col. 6" in col):
            total_col = col
            break
    # Fallback — last numeric-looking column with Total
    if total_col is None:
        for col in df.columns:
            if "Total" in col and "Number" in col:
                total_col = col
                break
    # Fallback 2 — for 2017 style
    if total_col is None:
        for col in df.columns:
            if "Total" in col and "Death" in col:
                total_col = col
                break

    if total_col is None:
        print(f"WARNING: Total column not found for year {year}")
        print(f"Columns: {list(df.columns)}")
        continue

    # Extract relevant columns
    df_clean = df[[name_col, total_col]].copy()
    df_clean.columns = ["State/UT", "Total Accidental Deaths"]

    # Standardize State/UT names
    df_clean["State/UT"] = df_clean["State/UT"].str.strip().str.upper().map(STATE_UT_NAMES)

    # Drop unmapped rows
    df_clean = df_clean.dropna(subset=["State/UT"])

    # Add year column
    df_clean["Year"] = year

    # Convert to numeric
    df_clean["Total Accidental Deaths"] = pd.to_numeric(df_clean["Total Accidental Deaths"], errors="coerce")

    # Merge Dadra & Daman and J&K & Ladakh if separate
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False)["Total Accidental Deaths"].sum()

    all_years_data.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

print("\nAll years processed!")

2015 — 29 States/UTs processed.
2016 — 35 States/UTs processed.
2017 — 0 States/UTs processed.
2018 — 35 States/UTs processed.


AttributeError: Can only use .str accessor with string values, not integer

In [114]:
# Debug - check 2017 and 2019 structure
for year in [2017, 2019]:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()
    print(f"\n{year} — Columns: {list(df.columns[:4])}")
    print(f"{year} — First 5 rows of first 4 columns:")
    print(df.iloc[:5, :4])
    print(f"{year} — Unique values in column 2 (Category): {df.iloc[:, 1].unique()}")


2017 — Columns: ['Sl. No. (Col.1)', 'Category (Col.2)', 'City (Col.3)', 'Total Number of Accidental Deaths - Forces of Nature (Col.4)']
2017 — First 5 rows of first 4 columns:
  Sl. No. (Col.1) Category (Col.2) City (Col.3)  Total Number of Accidental Deaths - Forces of Nature (Col.4)
0               1             City         Agra                                                  5           
1               2             City    Ahmedabad                                                  2           
2               3             City    Allahabad                                                  0           
3               4             City     Amritsar                                                 51           
4               5             City      Asansol                                                  1           
2017 — Unique values in column 2 (Category): <StringArray>
['City']
Length: 1, dtype: str

2019 — Columns: ['Category (Col. 1)', 'State/UT/City (Col. 2)', 'Number 

In [115]:
# Reload 2017 file and check structure of 2017 and 2019
for year in [2017, 2019]:
    file_path = os.path.join(BASE_PATH, str(year), f"ADSI_{year}_Table_1.2.csv")
    df = pd.read_csv(file_path, encoding='utf-8')
    print(f"\n{year} — Columns: {list(df.columns[:4])}")
    print(f"{year} — First 5 rows of first 4 columns:")
    print(df.iloc[:5, :4])
    print(f"{year} — Unique values in column 1 (Category): {df.iloc[:, 1].unique()}")


2017 — Columns: ['Sl. No.', 'Category', 'State/UT/City', 'Deaths due to Forces of Nature - 2016']
2017 — First 5 rows of first 4 columns:
  Sl. No. Category      State/UT/City  Deaths due to Forces of Nature - 2016
0       1    State     Andhra Pradesh                                    426
1       2    State  Arunachal Pradesh                                     47
2       3    State              Assam                                    544
3       4    State              Bihar                                   1057
4       5    State       Chhattisgarh                                    305
2017 — Unique values in column 1 (Category): <StringArray>
['State', 'Union Territory', 'Total (All India)', 'City']
Length: 4, dtype: str

2019 — Columns: ['Category (Col. 1)', 'State/UT/City (Col. 2)', 'Number of Accidental Deaths - Forces of Nature (Col. 3)', 'Number of Accidental Deaths - Other Causes (Col. 4)']
2019 — First 5 rows of first 4 columns:
  Category (Col. 1) State/UT/City (Col. 2

In [116]:
# Standard State/UT names mapping
STATE_UT_NAMES = {
    # States
    "ANDHRA PRADESH": "Andhra Pradesh",
    "ARUNACHAL PRADESH": "Arunachal Pradesh",
    "ASSAM": "Assam",
    "BIHAR": "Bihar",
    "CHHATTISGARH": "Chhattisgarh",
    "GOA": "Goa",
    "GUJARAT": "Gujarat",
    "HARYANA": "Haryana",
    "HIMACHAL PRADESH": "Himachal Pradesh",
    "JHARKHAND": "Jharkhand",
    "KARNATAKA": "Karnataka",
    "KERALA": "Kerala",
    "MADHYA PRADESH": "Madhya Pradesh",
    "MAHARASHTRA": "Maharashtra",
    "MANIPUR": "Manipur",
    "MEGHALAYA": "Meghalaya",
    "MIZORAM": "Mizoram",
    "NAGALAND": "Nagaland",
    "ODISHA": "Odisha",
    "PUNJAB": "Punjab",
    "RAJASTHAN": "Rajasthan",
    "SIKKIM": "Sikkim",
    "TAMIL NADU": "Tamil Nadu",
    "TELANGANA": "Telangana",
    "TRIPURA": "Tripura",
    "UTTAR PRADESH": "Uttar Pradesh",
    "UTTARAKHAND": "Uttarakhand",
    "WEST BENGAL": "West Bengal",
    # UTs - full forms
    "ANDAMAN AND NICOBAR ISLANDS": "Andaman and Nicobar Islands",
    "ANDAMAN & NICOBAR ISLANDS": "Andaman and Nicobar Islands",
    "A & N ISLANDS": "Andaman and Nicobar Islands",
    "A & NISLANDS": "Andaman and Nicobar Islands",
    "CHANDIGARH": "Chandigarh",
    "DADRA AND NAGAR HAVELI": "Dadra and Nagar Haveli and Daman and Diu",
    "DAMAN AND DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "DADRA AND NAGAR HAVELI AND DAMAN AND DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "DADRA & NAGAR HAVELI & DAMAN & DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "D & N HAVELI": "Dadra and Nagar Haveli and Daman and Diu",
    "DAMAN & DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "DELHI": "Delhi",
    "DELHI (UT)": "Delhi",
    "JAMMU AND KASHMIR": "Jammu and Kashmir",
    "JAMMU & KASHMIR": "Jammu and Kashmir",
    "LADAKH": "Jammu and Kashmir",
    "LAKSHADWEEP": "Lakshadweep",
    "PUDUCHERRY": "Puducherry",
    "PONDICHERRY": "Puducherry",
}

# Valid State/UT names list for 2019 type filtering
VALID_STATE_UT_KEYS = set(STATE_UT_NAMES.keys())

print("State/UT name mapping defined.")

State/UT name mapping defined.


In [118]:
# Debug 2017 and 2019
for year in [2017, 2019]:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()
    cat_col = df.columns[1]
    name_col = df.columns[2]
    print(f"\n{year}")
    print(f"cat_col: {cat_col}")
    print(f"name_col: {name_col}")
    print(f"cat_col unique values: {df[cat_col].unique()[:5]}")
    print(f"name_col dtype: {df[name_col].dtype}")
    print(df.iloc[:3, :4])


2017
cat_col: Category (Col.2)
name_col: City (Col.3)
cat_col unique values: <StringArray>
['City']
Length: 1, dtype: str
name_col dtype: str
  Sl. No. (Col.1) Category (Col.2) City (Col.3)  Total Number of Accidental Deaths - Forces of Nature (Col.4)
0               1             City         Agra                                                  5           
1               2             City    Ahmedabad                                                  2           
2               3             City    Allahabad                                                  0           

2019
cat_col: State/UT/City (Col. 2)
name_col: Number of Accidental Deaths - Forces of Nature (Col. 3)
cat_col unique values: <StringArray>
['ANDHRA PRADESH', 'ARUNACHAL PRADESH', 'ASSAM', 'BIHAR', 'CHHATTISGARH']
Length: 5, dtype: str
name_col dtype: int64
  Category (Col. 1) State/UT/City (Col. 2)  Number of Accidental Deaths - Forces of Nature (Col. 3)  Number of Accidental Deaths - Other Causes (Col. 4)
0    

In [119]:
# Fix 1 — Reload 2017 file
file_path_2017 = os.path.join(BASE_PATH, "2017", "ADSI_2017_Table_1.2.csv")
df_2017 = pd.read_csv(file_path_2017, encoding='utf-8')
all_files[2017]["Table_1.2"]["df"] = df_2017
print(f"2017 reloaded — Unique Category values: {df_2017.iloc[:, 1].unique()}")

# Fix 2 — Check 2019 column positions
df_2019 = all_files[2019]["Table_1.2"]["df"].copy()
df_2019.columns = df_2019.columns.str.strip()
print(f"\n2019 — col[0]: {df_2019.columns[0]}")
print(f"2019 — col[1]: {df_2019.columns[1]}")
print(f"2019 — col[0] unique (first 5): {df_2019.iloc[:, 0].unique()[:5]}")

2017 reloaded — Unique Category values: <StringArray>
['State', 'Union Territory', 'Total (All India)', 'City']
Length: 4, dtype: str

2019 — col[0]: Category (Col. 1)
2019 — col[1]: State/UT/City (Col. 2)
2019 — col[0] unique (first 5): <StringArray>
['State', 'UT', 'TOTAL (ALL INDIA)', 'City']
Length: 4, dtype: str


In [120]:
# Clean Table_1.2 - Total Accidental Deaths
all_years_data = []

for year in YEARS:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()

    # Detect column positions — some years have category in col[0], others in col[1]
    if df.iloc[:, 0].str.strip().str.upper().isin(["STATE", "UT", "UNION TERRITORY"]).any():
        cat_col = df.columns[0]
        name_col = df.columns[1]
    else:
        cat_col = df.columns[1]
        name_col = df.columns[2]

    # Filter only State and UT rows
    df = df[df[cat_col].str.strip().str.upper().isin(["STATE", "UT", "UNION TERRITORY"])]

    # Find total deaths column dynamically
    total_col = None
    for col in df.columns:
        if "Total" in col and ("Col. 5" in col or "Col.5" in col or "Col. 6" in col):
            total_col = col
            break
    if total_col is None:
        for col in df.columns:
            if "Total" in col and "Number" in col:
                total_col = col
                break
    if total_col is None:
        for col in df.columns:
            if "Total" in col and "Death" in col:
                total_col = col
                break

    if total_col is None:
        print(f"WARNING: Total column not found for year {year}")
        print(f"Columns: {list(df.columns)}")
        continue

    # Extract relevant columns
    df_clean = df[[name_col, total_col]].copy()
    df_clean.columns = ["State/UT", "Total Accidental Deaths"]

    # Standardize State/UT names
    df_clean["State/UT"] = df_clean["State/UT"].str.strip().str.upper().map(STATE_UT_NAMES)

    # Drop unmapped rows
    df_clean = df_clean.dropna(subset=["State/UT"])

    # Add year column
    df_clean["Year"] = year

    # Convert to numeric
    df_clean["Total Accidental Deaths"] = pd.to_numeric(df_clean["Total Accidental Deaths"], errors="coerce")

    # Merge Dadra & Daman and J&K & Ladakh if separate
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False)["Total Accidental Deaths"].sum()

    all_years_data.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

print("\nAll years processed!")

2015 — 29 States/UTs processed.
2016 — 0 States/UTs processed.
2017 — 0 States/UTs processed.
2018 — 0 States/UTs processed.
2019 — 35 States/UTs processed.
2020 — 28 States/UTs processed.
2021 — 35 States/UTs processed.
2022 — 28 States/UTs processed.

All years processed!


In [121]:
# Debug column positions for all years
for year in YEARS:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()
    print(f"\n{year}")
    print(f"col[0]: {df.columns[0]} — unique (first 3): {df.iloc[:, 0].unique()[:3]}")
    print(f"col[1]: {df.columns[1]} — unique (first 3): {df.iloc[:, 1].unique()[:3]}")


2015
col[0]: Sl. No. (Col.1) — unique (first 3): <StringArray>
['1', '2', '3']
Length: 3, dtype: str
col[1]: Category (Col.2) — unique (first 3): <StringArray>
['State', 'TOTAL (STATES)', 'Union Territories']
Length: 3, dtype: str

2016
col[0]: Sl. No. (Col.1) — unique (first 3): <StringArray>
['1', '2', '3']
Length: 3, dtype: str
col[1]: Category (Col.2) — unique (first 3): <StringArray>
['State', 'Union Territory', 'Total (All India)']
Length: 3, dtype: str

2017
col[0]: Sl. No. — unique (first 3): <StringArray>
['1', '2', '3']
Length: 3, dtype: str
col[1]: Category — unique (first 3): <StringArray>
['State', 'Union Territory', 'Total (All India)']
Length: 3, dtype: str

2018
col[0]: Sl. No. - (Col.1) — unique (first 3): <StringArray>
['1', '2', '3']
Length: 3, dtype: str
col[1]: Category - (Col.2) — unique (first 3): <StringArray>
['State', 'Union Territory', 'Total (All India)']
Length: 3, dtype: str

2019
col[0]: Category (Col. 1) — unique (first 3): <StringArray>
['State', 'UT',

In [122]:
# Clean Table_1.2 - Total Accidental Deaths
all_years_data = []

# Category values that mean State or UT
VALID_CATEGORIES = ["STATE", "UT", "UNION TERRITORY", "UNION TERRITORIES"]

for year in YEARS:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()

    # Detect column positions
    if df.iloc[:, 0].str.strip().str.upper().isin(VALID_CATEGORIES).any():
        # 2019, 2021 type
        cat_col = df.columns[0]
        name_col = df.columns[1]
    else:
        # 2015, 2016, 2017, 2018, 2020, 2022 type
        cat_col = df.columns[1]
        name_col = df.columns[2]

    # Filter only State and UT rows
    df = df[df[cat_col].str.strip().str.upper().isin(VALID_CATEGORIES)]

    # Find total deaths column dynamically
    total_col = None
    for col in df.columns:
        if "Total" in col and ("Col. 5" in col or "Col.5" in col or "Col. 6" in col):
            total_col = col
            break
    if total_col is None:
        for col in df.columns:
            if "Total" in col and "Number" in col:
                total_col = col
                break
    if total_col is None:
        for col in df.columns:
            if "Total" in col and "Death" in col:
                total_col = col
                break

    if total_col is None:
        print(f"WARNING: Total column not found for year {year}")
        print(f"Columns: {list(df.columns)}")
        continue

    # Extract relevant columns
    df_clean = df[[name_col, total_col]].copy()
    df_clean.columns = ["State/UT", "Total Accidental Deaths"]

    # Standardize State/UT names
    df_clean["State/UT"] = df_clean["State/UT"].str.strip().str.upper().map(STATE_UT_NAMES)

    # Drop unmapped rows
    df_clean = df_clean.dropna(subset=["State/UT"])

    # Add year column
    df_clean["Year"] = year

    # Convert to numeric
    df_clean["Total Accidental Deaths"] = pd.to_numeric(df_clean["Total Accidental Deaths"], errors="coerce")

    # Merge Dadra & Daman and J&K & Ladakh if separate
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False)["Total Accidental Deaths"].sum()

    all_years_data.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

print("\nAll years processed!")

2015 — 35 States/UTs processed.
2016 — 0 States/UTs processed.
2017 — 0 States/UTs processed.
2018 — 0 States/UTs processed.
2019 — 35 States/UTs processed.
2020 — 34 States/UTs processed.
2021 — 35 States/UTs processed.
2022 — 34 States/UTs processed.

All years processed!


In [123]:
# Debug total column for 2016, 2017, 2018
for year in [2016, 2017, 2018]:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()
    print(f"\n{year} — All columns:")
    for col in df.columns:
        print(f"  {col}")


2016 — All columns:
  Sl. No. (Col.1)
  Category (Col.2)
  State/UT (Col.3)
  Total Number of Accidental Deaths - Forces of Nature (Col.4)
  Total Number of Accidental Deaths - Other Causes (Col.5)
  Total Number of Accidental Deaths - Total (Col.6)
  Percentage Share in Total Deaths (Col.7)
  Projected Mid-Year Population* (In Lakh+) (Col.8)
  Rate of Accidental Deaths (Col.9) = (Col.6/Col.8)

2017 — All columns:
  Sl. No.
  Category
  State/UT/City
  Deaths due to Forces of Nature - 2016
  Deaths due to Forces of Nature - 2017
  Deaths due to Forces of Nature - % Variation
  Deaths due to Other Causes - 2016
  Deaths due to Other Causes - 2017
  Deaths due to Other Causes - % Variation
  Total Accidental Deaths - 2016
  Total Accidental Deaths - 2017
  Total Accidental Deaths - % Variation

2018 — All columns:
  Sl. No. - (Col.1)
  Category - (Col.2)
  State/UT - (Col.3)
  Total Number of Accidental Deaths - Forces of Nature - (Col.4)
  Total Number of Accidental Deaths - Other Caus

In [124]:
# Clean Table_1.2 - Total Accidental Deaths
all_years_data = []

VALID_CATEGORIES = ["STATE", "UT", "UNION TERRITORY", "UNION TERRITORIES"]

def find_total_col(df, year):
    for col in df.columns:
        # 2017 specific
        if f"Total Accidental Deaths - {year}" in col:
            return col
    for col in df.columns:
        if "Total" in col and ("Col. 5" in col or "Col.5" in col):
            return col
    for col in df.columns:
        if "Total" in col and ("Col. 6" in col or "Col.6" in col):
            return col
    for col in df.columns:
        if "Total" in col and "Number" in col:
            return col
    for col in df.columns:
        if "Total" in col and "Death" in col:
            return col
    return None

for year in YEARS:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()

    # Detect column positions
    if df.iloc[:, 0].str.strip().str.upper().isin(VALID_CATEGORIES).any():
        cat_col = df.columns[0]
        name_col = df.columns[1]
    else:
        cat_col = df.columns[1]
        name_col = df.columns[2]

    # Filter only State and UT rows
    df = df[df[cat_col].str.strip().str.upper().isin(VALID_CATEGORIES)]

    # Find total deaths column
    total_col = find_total_col(df, year)

    if total_col is None:
        print(f"WARNING: Total column not found for year {year}")
        continue

    # Extract relevant columns
    df_clean = df[[name_col, total_col]].copy()
    df_clean.columns = ["State/UT", "Total Accidental Deaths"]

    # Standardize State/UT names
    df_clean["State/UT"] = df_clean["State/UT"].str.strip().str.upper().map(STATE_UT_NAMES)

    # Drop unmapped rows
    df_clean = df_clean.dropna(subset=["State/UT"])

    # Add year column
    df_clean["Year"] = year

    # Convert to numeric
    df_clean["Total Accidental Deaths"] = pd.to_numeric(df_clean["Total Accidental Deaths"], errors="coerce")

    # Merge Dadra & Daman and J&K & Ladakh if separate
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False)["Total Accidental Deaths"].sum()

    all_years_data.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

print("\nAll years processed!")

2015 — 35 States/UTs processed.
2016 — 0 States/UTs processed.
2017 — 0 States/UTs processed.
2018 — 0 States/UTs processed.
2019 — 35 States/UTs processed.
2020 — 34 States/UTs processed.
2021 — 35 States/UTs processed.
2022 — 34 States/UTs processed.

All years processed!


In [125]:
# Debug 2016, 2017, 2018 step by step
for year in [2016, 2017, 2018]:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()
    
    # Check column detection
    if df.iloc[:, 0].str.strip().str.upper().isin(VALID_CATEGORIES).any():
        cat_col = df.columns[0]
        name_col = df.columns[1]
    else:
        cat_col = df.columns[1]
        name_col = df.columns[2]
    
    print(f"\n{year}")
    print(f"cat_col: {cat_col}")
    print(f"name_col: {name_col}")
    
    # Check after filter
    df_filtered = df[df[cat_col].str.strip().str.upper().isin(VALID_CATEGORIES)]
    print(f"Rows after filter: {len(df_filtered)}")
    print(f"Unique cat values: {df[cat_col].str.strip().str.upper().unique()[:5]}")
    
    # Check total col
    total_col = find_total_col(df_filtered, year)
    print(f"total_col found: {total_col}")
    
    # Check name mapping
    if len(df_filtered) > 0:
        sample = df_filtered[name_col].str.strip().str.upper().head(3)
        print(f"Sample names: {list(sample)}")


2016
cat_col: Sl. No. (Col.1)
name_col: Category (Col.2)
Rows after filter: 2
Unique cat values: <StringArray>
['1', '2', '3', '4', '5']
Length: 5, dtype: str
total_col found: Total Number of Accidental Deaths - Other Causes (Col.5)
Sample names: ['STATE', 'UNION TERRITORY']

2017
cat_col: Sl. No.
name_col: Category
Rows after filter: 2
Unique cat values: <StringArray>
['1', '2', '3', '4', '5']
Length: 5, dtype: str
total_col found: Total Accidental Deaths - 2017
Sample names: ['STATE', 'UNION TERRITORY']

2018
cat_col: Sl. No. - (Col.1)
name_col: Category - (Col.2)
Rows after filter: 2
Unique cat values: <StringArray>
['1', '2', '3', '4', '5']
Length: 5, dtype: str
total_col found: Total Number of Accidental Deaths - Other Causes - (Col.5)
Sample names: ['STATE', 'UNION TERRITORY']


In [135]:
# Clean Table_1.2 - Total Accidental Deaths
all_years_data = []

VALID_CATEGORIES = ["STATE", "UT", "UNION TERRITORY", "UNION TERRITORIES"]

def find_total_col(df, year):
    # 2017 specific
    for col in df.columns:
        if f"Total Accidental Deaths - {year}" in col:
            return col
    # Col.6 priority
    for col in df.columns:
        if "Total" in col and ("Col. 6" in col or "Col.6" in col):
            return col
    # Col.5 only if nothing else
    for col in df.columns:
        if "Total" in col and ("Col. 5" in col or "Col.5" in col):
            return col
    for col in df.columns:
        if "Total" in col and "Number" in col:
            return col
    for col in df.columns:
        if "Total" in col and "Death" in col:
            return col
    return None

def detect_cols(df):
    # Check col[1] first — most years have Category there
    if df.iloc[:, 1].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES).any():
        return df.columns[1], df.columns[2]
    # Check col[0] — 2019, 2021 type
    elif df.iloc[:, 0].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES).any():
        return df.columns[0], df.columns[1]
    else:
        return df.columns[1], df.columns[2]

for year in YEARS:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()

    cat_col, name_col = detect_cols(df)

    # Filter only State and UT rows
    df = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]

    # Find total deaths column
    total_col = find_total_col(df, year)

    if total_col is None:
        print(f"WARNING: Total column not found for year {year}")
        continue

    # Extract relevant columns
    df_clean = df[[name_col, total_col]].copy()
    df_clean.columns = ["State/UT", "Total Accidental Deaths"]

    # Standardize State/UT names
    df_clean["State/UT"] = df_clean["State/UT"].astype(str).str.strip().str.upper().map(STATE_UT_NAMES)

    # Drop unmapped rows
    df_clean = df_clean.dropna(subset=["State/UT"])

    # Add year column
    df_clean["Year"] = year

    # Convert to numeric
    df_clean["Total Accidental Deaths"] = pd.to_numeric(df_clean["Total Accidental Deaths"], errors="coerce")

    # Merge Dadra & Daman and J&K & Ladakh if separate
    df_clean = df_clean.groupby(["State/UT", "Year"], as_index=False)["Total Accidental Deaths"].sum()

    all_years_data.append(df_clean)
    print(f"{year} — {len(df_clean)} States/UTs processed.")

print("\nAll years processed!")

2015 — 35 States/UTs processed.
2016 — 35 States/UTs processed.
2017 — 35 States/UTs processed.
2018 — 35 States/UTs processed.
2019 — 35 States/UTs processed.
2020 — 35 States/UTs processed.
2021 — 35 States/UTs processed.
2022 — 35 States/UTs processed.

All years processed!


In [127]:
# Debug 2020 and 2022 — which State/UT is missing
all_35 = set(all_years_data[0]["State/UT"].unique())  # 2015 as reference

for i, year in enumerate(YEARS):
    if len(all_years_data[i]) == 34:
        present = set(all_years_data[i]["State/UT"].unique())
        missing = all_35 - present
        print(f"{year} — Missing: {missing}")

2020 — Missing: {'Dadra and Nagar Haveli and Daman and Diu'}
2022 — Missing: {'Andaman and Nicobar Islands'}


In [128]:
# Debug 2020 and 2022 — check raw names
for year in [2020, 2022]:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()
    cat_col, name_col = detect_cols(df)
    df_filtered = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]
    print(f"\n{year} — Raw State/UT names:")
    print(df_filtered[name_col].str.strip().tolist())


2020 — Raw State/UT names:
['Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chhattisgarh', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal', 'A & N Islands', 'Chandigarh', 'D & N Haveli and Daman & Diu', 'Delhi (UT)', 'Jammu & Kashmir', 'Ladakh', 'Lakshadweep', 'Puducherry']

2022 — Raw State/UT names:
['Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chhattisgarh', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal', 'Andamand and Nicobar Islands', 'Chandigarh', 'Dadra and Nagar Haveli and D

In [131]:
# Debug — check which States/UTs are missing for each year
reference = set(['Andhra Pradesh', 'Arunachal Pradesh', 'Assam', 'Bihar', 'Chhattisgarh', 'Goa', 'Gujarat', 'Haryana', 'Himachal Pradesh', 'Jharkhand', 'Karnataka', 'Kerala', 'Madhya Pradesh', 'Maharashtra', 'Manipur', 'Meghalaya', 'Mizoram', 'Nagaland', 'Odisha', 'Punjab', 'Rajasthan', 'Sikkim', 'Tamil Nadu', 'Telangana', 'Tripura', 'Uttar Pradesh', 'Uttarakhand', 'West Bengal', 'Andaman and Nicobar Islands', 'Chandigarh', 'Dadra and Nagar Haveli and Daman and Diu', 'Delhi', 'Jammu and Kashmir', 'Lakshadweep', 'Puducherry'])

for i, year in enumerate(YEARS):
    present = set(all_years_data[i]["State/UT"].unique())
    missing = reference - present
    extra = present - reference
    print(f"{year} — Missing: {missing}")
    print(f"{year} — Extra: {extra}")
    print()

2015 — Missing: {'Delhi', 'Dadra and Nagar Haveli and Daman and Diu', 'Andaman and Nicobar Islands'}
2015 — Extra: set()

2016 — Missing: {'Delhi', 'Dadra and Nagar Haveli and Daman and Diu', 'Andaman and Nicobar Islands'}
2016 — Extra: set()

2017 — Missing: {'Delhi', 'Dadra and Nagar Haveli and Daman and Diu', 'Andaman and Nicobar Islands'}
2017 — Extra: set()

2018 — Missing: {'Delhi', 'Dadra and Nagar Haveli and Daman and Diu', 'Andaman and Nicobar Islands'}
2018 — Extra: set()

2019 — Missing: {'Delhi', 'Dadra and Nagar Haveli and Daman and Diu', 'Andaman and Nicobar Islands'}
2019 — Extra: set()

2020 — Missing: {'Delhi', 'Andaman and Nicobar Islands'}
2020 — Extra: set()

2021 — Missing: set()
2021 — Extra: set()

2022 — Missing: set()
2022 — Extra: set()



In [132]:
# Debug — check raw names for missing entries in 2015, 2019, 2020
search_terms = ["DELHI", "ANDAMAN", "DADRA", "DAMAN", "HAVELI"]

for year in [2015, 2019, 2020]:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()
    cat_col, name_col = detect_cols(df)
    df_filtered = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]
    names = df_filtered[name_col].astype(str).str.strip().tolist()
    matched = [n for n in names if any(term in n.upper() for term in search_terms)]
    print(f"\n{year} — Relevant raw names: {matched}")


2015 — Relevant raw names: ['D & N HAVELI', 'DAMAN & DIU', 'DELHI (UT)']

2019 — Relevant raw names: ['D & N HAVELI', 'DAMAN & DIU', 'DELHI (UT)']

2020 — Relevant raw names: ['D & N Haveli and Daman & Diu', 'Delhi (UT)']


In [133]:
# Debug — check Andaman raw names for 2015-2019
for year in [2015, 2016, 2017, 2018, 2019]:
    df = all_files[year]["Table_1.2"]["df"].copy()
    df.columns = df.columns.str.strip()
    cat_col, name_col = detect_cols(df)
    df_filtered = df[df[cat_col].astype(str).str.strip().str.upper().isin(VALID_CATEGORIES)]
    names = df_filtered[name_col].astype(str).str.strip().tolist()
    matched = [n for n in names if "ANDAMAN" in n.upper() or "A &" in n.upper()]
    print(f"{year} — Andaman raw name: {matched}")

2015 — Andaman raw name: ['A & N ISLANDS']
2016 — Andaman raw name: ['A & N Islands']
2017 — Andaman raw name: ['A & N Islands']
2018 — Andaman raw name: ['A & N Islands']
2019 — Andaman raw name: ['A & NISLANDS']


In [134]:
# Updated STATE_UT_NAMES mapping
STATE_UT_NAMES = {
    # States
    "ANDHRA PRADESH": "Andhra Pradesh",
    "ARUNACHAL PRADESH": "Arunachal Pradesh",
    "ASSAM": "Assam",
    "BIHAR": "Bihar",
    "CHHATTISGARH": "Chhattisgarh",
    "GOA": "Goa",
    "GUJARAT": "Gujarat",
    "HARYANA": "Haryana",
    "HIMACHAL PRADESH": "Himachal Pradesh",
    "JHARKHAND": "Jharkhand",
    "KARNATAKA": "Karnataka",
    "KERALA": "Kerala",
    "MADHYA PRADESH": "Madhya Pradesh",
    "MAHARASHTRA": "Maharashtra",
    "MANIPUR": "Manipur",
    "MEGHALAYA": "Meghalaya",
    "MIZORAM": "Mizoram",
    "NAGALAND": "Nagaland",
    "ODISHA": "Odisha",
    "PUNJAB": "Punjab",
    "RAJASTHAN": "Rajasthan",
    "SIKKIM": "Sikkim",
    "TAMIL NADU": "Tamil Nadu",
    "TELANGANA": "Telangana",
    "TRIPURA": "Tripura",
    "UTTAR PRADESH": "Uttar Pradesh",
    "UTTARAKHAND": "Uttarakhand",
    "WEST BENGAL": "West Bengal",
    # UTs
    "ANDAMAN AND NICOBAR ISLANDS": "Andaman and Nicobar Islands",
    "ANDAMAN & NICOBAR ISLANDS": "Andaman and Nicobar Islands",
    "A & N ISLANDS": "Andaman and Nicobar Islands",
    "A & NISLANDS": "Andaman and Nicobar Islands",
    "ANDAMAND AND NICOBAR ISLANDS": "Andaman and Nicobar Islands",
    "CHANDIGARH": "Chandigarh",
    "DADRA AND NAGAR HAVELI": "Dadra and Nagar Haveli and Daman and Diu",
    "DAMAN AND DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "DADRA AND NAGAR HAVELI AND DAMAN AND DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "DADRA & NAGAR HAVELI & DAMAN & DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "D & N HAVELI": "Dadra and Nagar Haveli and Daman and Diu",
    "DAMAN & DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "D & N HAVELI AND DAMAN & DIU": "Dadra and Nagar Haveli and Daman and Diu",
    "DELHI": "Delhi",
    "DELHI (UT)": "Delhi",
    "JAMMU AND KASHMIR": "Jammu and Kashmir",
    "JAMMU & KASHMIR": "Jammu and Kashmir",
    "LADAKH": "Jammu and Kashmir",
    "LAKSHADWEEP": "Lakshadweep",
    "PUDUCHERRY": "Puducherry",
    "PONDICHERRY": "Puducherry",
}

VALID_STATE_UT_KEYS = set(STATE_UT_NAMES.keys())
print("State/UT name mapping updated.")

State/UT name mapping updated.


In [136]:
# Merge all years and add S.No.
final_df = pd.concat(all_years_data, ignore_index=True)

# Sort by Year then alphabetically by State/UT
final_df = final_df.sort_values(["Year", "State/UT"]).reset_index(drop=True)

# Add continuous S.No.
final_df.insert(0, "S.No.", range(1, len(final_df) + 1))

# Reorder columns
final_df = final_df[["S.No.", "State/UT", "Year", "Total Accidental Deaths"]]

# Save to processed folder
final_df.to_csv("../data/processed/total_accidental_deaths_2015_2022.csv", index=False)

print(f"Total rows: {len(final_df)}")
print(f"Years: {final_df['Year'].unique()}")
print(f"\nFirst 10 rows:")
print(final_df.head(10))

Total rows: 280
Years: [2015 2016 2017 2018 2019 2020 2021 2022]

First 10 rows:
   S.No.                                  State/UT  Year  Total Accidental Deaths
0      1               Andaman and Nicobar Islands  2015                     91.0
1      2                            Andhra Pradesh  2015                   1321.0
2      3                         Arunachal Pradesh  2015                     23.0
3      4                                     Assam  2015                    185.0
4      5                                     Bihar  2015                    481.0
5      6                                Chandigarh  2015                     67.0
6      7                              Chhattisgarh  2015                   4679.0
7      8  Dadra and Nagar Haveli and Daman and Diu  2015                     42.0
8      9                                     Delhi  2015                    393.0
9     10                                       Goa  2015                     54.0
